In [10]:
import os
import pandas as pd

DATA_DIR = "../data/processed"

files_to_convert = [
    "SeoulBikeRental_20260401_7days_processed.csv",
    "SeoulBikeRental_20260401_7days_station_categories.csv",
    # 필요하면 다른 CSV 파일도 여기에 추가
    # "SeoulBikeStationMaster_processed.csv",
]

for filename in files_to_convert:
    csv_path = f"{DATA_DIR}/{filename}"
    parquet_path = csv_path.replace(".csv", ".parquet")

    # 인코딩 문제 방지를 위해 utf-8-sig로 명시해서 읽기 (cp949인 경우 아래 except가 처리)
    try:
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding="cp949")

    # parquet으로 저장 (snappy 압축, pandas 기본값)
    df.to_parquet(parquet_path, index=False)

    # 용량 비교 출력
    csv_size = os.path.getsize(csv_path) / 1024**2
    parquet_size = os.path.getsize(parquet_path) / 1024**2
    print(f"[{filename}]")
    print(f"  CSV     : {csv_size:.1f} MB")
    print(f"  Parquet : {parquet_size:.1f} MB")
    print(f"  절감률  : {(1 - parquet_size / csv_size) * 100:.1f}%\n")

[SeoulBikeRental_20260401_7days_processed.csv]
  CSV     : 125.4 MB
  Parquet : 20.3 MB
  절감률  : 83.8%

[SeoulBikeRental_20260401_7days_station_categories.csv]
  CSV     : 144.4 MB
  Parquet : 21.1 MB
  절감률  : 85.4%



In [9]:
import chardet
import pandas as pd
from huggingface_hub import HfApi


files_to_process = [
    "SeoulBikeRental_20260401_7days_processed.csv",
    "SeoulBikeRental_20260401_7days_station_categories.csv",
]
DATA_DIR = "../data/processed"

for filename in files_to_process:
    file_path = f"{DATA_DIR}/{filename}"

    # 1. 인코딩 감지
    with open(file_path, "rb") as f:
        raw = f.read(50000)  # 앞부분만 읽어서 판별 (전체 읽으면 대용량이라 느림)
        detected = chardet.detect(raw)
    encoding = detected["encoding"]
    confidence = detected["confidence"]
    print(f"[{filename}] 감지된 인코딩: {encoding} (신뢰도: {confidence:.2f})")

    # 2. 감지된 인코딩으로 읽기 (실패하면 cp949로 재시도)
    try:
        df = pd.read_csv(file_path, encoding=encoding)
    except (UnicodeDecodeError, LookupError):
        print(f"  → {encoding}로 읽기 실패, cp949로 재시도")
        df = pd.read_csv(file_path, encoding="cp949")

    # 3. UTF-8(BOM 포함)로 다시 저장 → 원본 파일을 덮어씀
    df.to_csv(file_path, index=False, encoding="utf-8-sig")
    print(f"  → UTF-8(BOM)로 재저장 완료: {file_path}")

# 4. Hugging Face에 업로드
api = HfApi()

for filename in files_to_process:
    file_path = f"{DATA_DIR}/{filename}"
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo=filename,
        repo_id="e-un000/SeoulBikeRental",
        repo_type="dataset",
    )
    print(f"업로드 완료: {filename}")

print("\n모든 파일 처리 및 업로드가 완료되었습니다.")

[SeoulBikeRental_20260401_7days_processed.csv] 감지된 인코딩: UTF-8-SIG (신뢰도: 1.00)
  → UTF-8(BOM)로 재저장 완료: ../data/processed/SeoulBikeRental_20260401_7days_processed.csv
[SeoulBikeRental_20260401_7days_station_categories.csv] 감지된 인코딩: UTF-8-SIG (신뢰도: 1.00)
  → UTF-8(BOM)로 재저장 완료: ../data/processed/SeoulBikeRental_20260401_7days_station_categories.csv


Processing Files (1 / 1): 100%|██████████|  132MB /  132MB, 12.0MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


업로드 완료: SeoulBikeRental_20260401_7days_processed.csv


Processing Files (1 / 1): 100%|██████████|  151MB /  151MB, 13.8MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


업로드 완료: SeoulBikeRental_20260401_7days_station_categories.csv

모든 파일 처리 및 업로드가 완료되었습니다.


In [4]:
import sys
sys.path.append("..")

from src.data.loader import load_csv
from src.data.preprocessing import clean_station_df
from src.analysis.analysis import add_imbalance

raw_df = load_csv("SeoulBikeStationUseInfo_2601to2606.csv")
df = clean_station_df(raw_df)
add_imbalance(df)


# df.describe().T

,district,stationName,statMn,rentCnt,rtnCnt,statMn_dt,imbalance
0,강남구,2301. 현대고등학교 건너편,202601,154,183,2026-01-01,-29
1,강남구,2302. 교보타워 버스정류장(신논현역 3번출구 후면),202601,467,492,2026-01-01,-25
2,강남구,2303. 논현역 10번출구,202601,501,332,2026-01-01,169
3,강남구,2304. 대현그린타워,202601,79,44,2026-01-01,35
4,강남구,2305. MCM 본사 직영점 앞,202601,149,177,2026-01-01,-28
...,...,...,...,...,...,...,...
16619,중랑구,4842. 면목라온프라이빗 아파트,202606,1025,1065,2026-06-01,-40
16620,중랑구,4843. 봉화산역 4번 출구,202606,1088,1257,2026-06-01,-169
16621,중랑구,4845. 신내역금강펜테리움센트럴파크,202606,233,206,2026-06-01,27
16622,중랑구,4846. 상봉동양엔파트 앞,202606,985,989,2026-06-01,-4
